# 00 — Setup & Smoke Test

Run this first on **Colab Pro (GPU runtime)**. It installs deps, logs into Hugging Face, mounts Drive for artifacts, and verifies that **both Llama models load in 4-bit** and produce a chain-of-thought trace whose answer we can extract.

**Prerequisites:**
- Gated access approved for `meta-llama/Llama-3.1-8B-Instruct` and `meta-llama/Llama-3.2-1B-Instruct`.
- `HF_TOKEN` saved as a Colab secret (key icon, left sidebar).

In [ ]:
# Clone the repo (replace with your GitHub URL) and install the package + deps.
import os
REPO_URL = "https://github.com/<your-username>/math-distillation.git"
if not os.path.exists("math-distillation"):
    !git clone $REPO_URL
%cd math-distillation
!pip -q install -e .
!pip -q install -r requirements.txt

# Make the freshly cloned package importable in THIS kernel session without a
# restart. An editable install registers its path via a .pth file that Python
# only reads at interpreter startup, so the already-running kernel can't see it.
import sys
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
# Hugging Face login (reads the HF_TOKEN Colab secret).
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # falls back to interactive prompt off-Colab

In [ ]:
# Mount Drive and point all artifacts there (survives disconnects).
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MATHDISTILL_HOME"] = "/content/drive/MyDrive/math-distillation"
    os.makedirs(os.environ["MATHDISTILL_HOME"], exist_ok=True)
    print("Artifacts ->", os.environ["MATHDISTILL_HOME"])
except Exception as e:
    print("Not on Colab / Drive not mounted:", e)

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")

In [ ]:
# Smoke test: load each model in 4-bit, generate one CoT, extract the answer (expect 72).
import torch
from mathdistill.utils import load_config
from mathdistill.models import load_model_4bit, load_tokenizer, render_prompt
from mathdistill.prompts import build_messages
from mathdistill.answers import extract_pred_number

tcfg = load_config("configs/teacher_gen.yaml")
q = ("Natalia sold clips to 48 friends in April, and half as many in May. "
     "How many clips did she sell altogether?")

for model_id in [tcfg["model_id"], "meta-llama/Llama-3.2-1B-Instruct"]:
    print("=" * 60, "\nLoading", model_id)
    tok = load_tokenizer(model_id)
    model = load_model_4bit(model_id, tcfg["quantization"])
    enc = tok(render_prompt(tok, build_messages(q)), return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(text)
    print(">> extracted:", extract_pred_number(text), "(expected 72)")
    del model
    torch.cuda.empty_cache()